# Aggregation

In the previous notebook, we retrieved individual rows from the database. That is useful for inspection, but most questions the transport authority actually asks are about groups: "How many passengers per line?" "What is the busiest hour?" "Which incident type causes the longest delays?"

The explainer introduced aggregation as the operation that compresses many rows into a single summary value. We saw how it connects to what we already do with pivot tables and functions like `SUM`, `AVERAGE`, and `COUNT` in spreadsheets. Now we will write those same operations in SQL.

By the end of this notebook we will be able to:

- Use **aggregate functions** (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) to summarise data
- Group rows with **`GROUP BY`** on one or more columns
- Filter grouped results with **`HAVING`**
- Rename output columns with **`AS`**

---

## Setup

In [ ]:
%pip install -q jupysql

In [ ]:
%load_ext sql

In [ ]:
%sql sqlite:///../data/metro_transit.sqlite

---

## 1. Aggregate functions

An **aggregate function** takes a set of values and returns a single result. These are the SQL equivalents of the spreadsheet functions we already know.

| Function | Returns | Spreadsheet equivalent |
|----------|---------|----------------------|
| `COUNT(*)` | Number of rows | `COUNTA` |
| `COUNT(column)` | Number of non-NULL values in that column | `COUNTA` on a range |
| `SUM(column)` | Total of all values | `SUM` |
| `AVG(column)` | Mean of all values | `AVERAGE` |
| `MIN(column)` | Smallest value | `MIN` |
| `MAX(column)` | Largest value | `MAX` |

The `ridership` table contains passenger counts by station, date, and hour. Start by getting a high-level picture of the whole dataset:

In [ ]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(passenger_count) AS total_passengers,
    AVG(passenger_count) AS avg_passengers,
    MIN(passenger_count) AS min_passengers,
    MAX(passenger_count) AS max_passengers
FROM ridership;

Notice the use of `AS` to give each result a readable name. Without it, the column headers would show the raw expression (e.g. `SUM(passenger_count)`), which makes results harder to read and share.

### COUNT(*) vs COUNT(column)

There is an important distinction here. `COUNT(*)` counts every row, including rows where some columns are `NULL`. `COUNT(column)` counts only rows where that specific column has a value.

The `staff` table has `NULL` values in `station_id` for control room staff who are not assigned to a specific station. Compare the two counts:

In [ ]:
%%sql
SELECT
    COUNT(*) AS total_staff,
    COUNT(station_id) AS staff_with_station
FROM staff;

The difference between those two numbers tells us how many control room staff have no assigned station. This is a practical example of why `NULL` handling matters in databases, as the explainer discussed.

---

## 2. GROUP BY: "for each X, what is the Y?"

A single aggregate across the whole table gives us one number. That is a start, but the authority's questions are usually more specific: "How many stations *per line*?" "What is the average ridership *per hour*?"

The explainer framed this as the "for each X, what is the Y?" pattern. In a spreadsheet, we would build a pivot table. In SQL, we use `GROUP BY`. It splits the data into groups based on one or more columns, then applies the aggregate function to each group separately.

How many stations are on each line?

In [ ]:
%%sql
SELECT line, COUNT(*) AS station_count
FROM stations
GROUP BY line
ORDER BY station_count DESC;

Now a question about ridership patterns. For each hour of the day, what is the average passenger count across all stations and dates?

In [ ]:
%%sql
SELECT hour, ROUND(AVG(passenger_count), 1) AS avg_passengers
FROM ridership
GROUP BY hour
ORDER BY hour;

We should see higher averages during the morning rush (07:00–09:00) and evening rush (17:00–19:00). This is exactly the kind of summary the authority needs for capacity planning.

### GROUP BY multiple columns

We can group by more than one column. This gives one row per unique combination, like adding a second field to the Rows area of a pivot table.

The authority wants to see incident counts broken down by both type and severity:

In [ ]:
%%sql
SELECT incident_type, severity, COUNT(*) AS incident_count
FROM incidents
GROUP BY incident_type, severity
ORDER BY incident_type, severity;

---

## 3. HAVING: filtering after aggregation

We already know `WHERE` filters individual rows *before* grouping. But what if we want to filter the *results* of an aggregation? For example, the authority asks: "Which hours of the day have an average passenger count above 300?" We cannot use `WHERE` for this because the average does not exist until after the grouping is done.

`HAVING` filters groups *after* aggregation. It works on the summarised values.

In [ ]:
%%sql
SELECT hour, ROUND(AVG(passenger_count), 1) AS avg_passengers
FROM ridership
GROUP BY hour
HAVING avg_passengers > 300
ORDER BY avg_passengers DESC;

A common mistake is using `WHERE` to filter on an aggregate. This will produce an error:

```sql
-- WRONG: WHERE cannot reference aggregates
SELECT hour, AVG(passenger_count) AS avg_passengers
FROM ridership
WHERE AVG(passenger_count) > 300
GROUP BY hour;
```

The rule is straightforward: use `WHERE` for row-level filters (before grouping) and `HAVING` for group-level filters (after grouping).

### Combining WHERE and HAVING

We can use both in the same query. `WHERE` runs first to filter rows, then `GROUP BY` groups the remaining rows, then `HAVING` filters the groups.

Here is a realistic question: among weekday ridership only, which stations have total passengers above 50,000? This needs `WHERE` to exclude weekends and `HAVING` to filter on the total:

In [ ]:
%%sql
SELECT station_id, SUM(passenger_count) AS total_passengers
FROM ridership
WHERE strftime('%w', date) NOT IN ('0', '6')
GROUP BY station_id
HAVING total_passengers > 50000
ORDER BY total_passengers DESC
LIMIT 10;

The `strftime('%w', date)` function extracts the day of the week (0 = Sunday, 6 = Saturday). This is a SQLite-specific function for working with dates. Other database systems have their own date functions, but the principle is the same.

---

## 4. Clause order recap

When we combine everything, the clauses in a `SELECT` statement must appear in this order:

```sql
SELECT columns / aggregates
FROM table
WHERE row-level conditions
GROUP BY grouping columns
HAVING group-level conditions
ORDER BY sort columns
LIMIT n;
```

Not every query needs all of these, but when they appear, they must follow this sequence. It helps to think of it as a pipeline: start with all rows, filter them, group them, filter the groups, sort, then limit.

---

## Exercises

Time to apply what we have learned. Each exercise asks a question the transport authority might ask. Write the SQL to answer it.

### Exercise 1: Total passengers per line

Join the `ridership` table to the `stations` table on `station_id` and calculate the total `passenger_count` for each `line`. Order by total passengers descending.

Hint: you will need a `JOIN` here. If you have not used joins before, the syntax is:

```sql
SELECT s.line, SUM(r.passenger_count) AS total
FROM ridership r
JOIN stations s ON r.station_id = s.station_id
GROUP BY s.line
```

In [ ]:
%%sql
-- Your query here


### Exercise 2: Busiest hour of the day

Find the single hour with the highest total passenger count across all stations and dates. Return the hour and the total.

In [ ]:
%%sql
-- Your query here


### Exercise 3: Average delay by incident type

Calculate the average `delay_minutes` for each `incident_type`. Round to one decimal place and order by average delay descending.

In [ ]:
%%sql
-- Your query here


### Exercise 4: Stations with more than 15 incidents

Find all station_ids that have more than 15 incidents. Return the station_id and the incident count, ordered by count descending.

In [ ]:
%%sql
-- Your query here


---

## Summary

We can now summarise data at scale using SQL. In this notebook we:

- Used `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX` to compress rows into single values
- Understood the difference between `COUNT(*)` and `COUNT(column)` for handling `NULL` values
- Grouped rows with `GROUP BY` on single and multiple columns
- Filtered grouped results with `HAVING`
- Gave readable names to output columns with `AS`

These are the SQL equivalents of the pivot table operations we already know from spreadsheets. The difference is that SQL handles millions of rows without slowing down.

In the next notebook, we will use **joins** to combine data from multiple tables — answering questions that no single table can answer on its own.